# FER: capsule fusion with face geometry

## Requirements
Set up the environment once, in a terminal, before opening this notebook.

| | |
|---|---|
| OS | Linux, or Windows 11 with WSL2 (Ubuntu 22.04 or 24.04). TensorFlow has no GPU support on native Windows after version 2.10. |
| GPU | NVIDIA with 16 GB (planned: RTX 4080 Super). Under WSL2, install only the normal Windows NVIDIA driver, nothing inside Linux. |
| Python | 3.10, 3.11 or 3.12 (TensorFlow 2.19) |
| Disk | about 15 GB free: CUDA libraries ~3 GB, scratch images ~4 GB, outputs, downloaded model weights |
| Internet | needed on the first run: pip packages, ViT and VGG16 weights, MediaPipe and its landmark model |

```text
# pip requirements
tensorflow[and-cuda]==2.19.*      # TensorFlow with the CUDA 12 and cuDNN libraries (Linux)
tf-keras==2.19.*                  # Keras 2, which the Hugging Face ViT needs
transformers>=4.40,<5             # version 5 removed the TensorFlow models
opencv-python-headless            # image reading, CLAHE
pandas
scikit-learn
scipy                             # rotations and shifts in ImageDataGenerator
matplotlib
seaborn
pillow
ipykernel                         # lets VS Code / Jupyter use this environment as the notebook kernel
# Not listed on purpose: MediaPipe 1.0.1. The notebook installs it into its own folder, because it needs a
# newer numpy than TensorFlow 2.19 allows.
```
This list resolves on Linux for Python 3.10 and 3.12 (TensorFlow 2.19.1 with CUDA 12.5 and cuDNN 9.3). The
notebook was tested with Python 3.11, TensorFlow 2.19.1, tf-keras 2.19.0, transformers 4.57.6, numpy 2.1.3,
pandas 3.0.6, scikit-learn 1.9.1 and OpenCV 5.0, on a CPU.

```bash
# Windows only, once, in PowerShell as administrator, then reboot:
#   wsl --install -d Ubuntu-22.04
sudo apt update && sudo apt install -y python3-venv python3-pip
python3 -m venv ~/venvs/fer && source ~/venvs/fer/bin/activate
pip install --upgrade pip
pip install "tensorflow[and-cuda]==2.19.*" "tf-keras==2.19.*" "transformers>=4.40,<5" \
    opencv-python-headless pandas scikit-learn scipy matplotlib seaborn pillow ipykernel
python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"   # must list the GPU
```
Then open this notebook in VS Code (WSL extension) and pick `~/venvs/fer` as the kernel. The next cell installs
anything that is still missing.

In [ ]:
# Installs whatever from the requirements above is missing (quick when everything is already there).
# On Kaggle, TensorFlow and tf-keras come with the image, so only transformers is pinned.
import os
import subprocess
import sys

assert (3, 9) <= sys.version_info[:2] <= (3, 12), f"Python {sys.version.split()[0]}: TensorFlow 2.19 needs Python 3.9-3.12"
if os.path.isdir("/kaggle/input"):
    REQUIREMENTS = ["transformers>=4.40,<5"]
else:
    REQUIREMENTS = [
        "tensorflow[and-cuda]==2.19.*" if sys.platform.startswith("linux") else "tensorflow==2.19.*",
        "tf-keras==2.19.*", "transformers>=4.40,<5", "opencv-python-headless", "pandas", "scikit-learn",
        "scipy", "matplotlib", "seaborn", "pillow",
    ]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *REQUIREMENTS], check=True)
print("Requirements in place. If this cell installed or upgraded TensorFlow, restart the kernel before going on.")

In [ ]:
# Run this cell before anything imports TensorFlow (restart the kernel if TF was already imported).
# The HF TFViTModel is a tf-keras (Keras 2) model. Inside a Keras 3 model its weights are not tracked,
# so the ViT silently never trains. Running the whole notebook on tf-keras makes it a proper sub-layer.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import random
import numpy as np
import tensorflow as tf
import transformers

import tf_keras

# tf.keras has no __version__ under tf-keras, so compare the classes instead
assert tf.keras.Model is tf_keras.Model, (
    "tf.keras is not tf-keras (Keras 2); restart the kernel and run this cell before any TensorFlow import"
)
assert int(transformers.__version__.split(".")[0]) < 5, "transformers>=5 has no TensorFlow models"
print("TensorFlow:", tf.__version__, "| tf-keras:", tf_keras.__version__, "| transformers:", transformers.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU found. Training on a CPU would take days; on Windows, run this notebook inside WSL2.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## How to run the face-geometry ablation
The notebook trains one of two arms, chosen by `USE_GEOMETRY` in the next cell:
- `False`, **baseline**: image only (VGG16 + CBAM and ViT, fused by the capsule block).
- `True`, **geometry**: the same model plus a third capsule built from 34 facial-landmark measurements.

Run it once per arm with the same split and seed. Each run saves its per-image test predictions
(`predictions_<arm>_test.npz`, `predictions_<arm>_cross.npz`). The last section compares the two arms with
bootstrap confidence intervals once both files are present (in this run's output, or attached as an input).

Each run trains to the end in one go. It runs on a desktop or on Kaggle (where a run must finish within 12 hours). On a desktop (GPU training needs WSL2 on Windows), the folders are set in the
next cell; by default, next to this notebook:
```
<project folder>/
├── exp-tuning.ipynb
├── data/                                    you add this (read only)
│   ├── combined/combined/train/<7 classes>/   (combined/train/... also works)
│   ├── combined/combined/test/<7 classes>/
│   ├── emotion-detection-fer/test/<7 classes>/
│   └── baseline_run/                        optional: final weights of a finished baseline run (PRETRAINED_WEIGHTS)
└── outputs/                                 created: landmarks, predictions, final weights
~/fer_scratch/                               created: rebuilt images and MediaPipe (about 4 GB)
```
The class folders are `angry disgust fear happy neutral sad surprise` (cross-dataset: `angry disgusted fearful
happy neutral sad surprised`). The notebook also writes `extract_landmarks.py` next to itself.

In [ ]:
# =================================
# Experiment settings and paths
# =================================
import os
import glob

# Ablation arm: True = image + face geometry (third capsule); False = image only (the corrected baseline)
USE_GEOMETRY = True
VARIANT = "geometry" if USE_GEOMETRY else "baseline"

# Final weights of an earlier run of the same arm (TensorFlow checkpoint path without .index). When set,
# training is skipped and the model is only evaluated. Example, the corrected baseline from Kaggle:
# PRETRAINED_WEIGHTS = os.path.join("data", "baseline_run", "vgg16_cbam_vit_incremental_unfreeze_improved")
PRETRAINED_WEIGHTS = None

ON_KAGGLE = os.path.isdir("/kaggle/input")

# ---- Desktop folders: edit these if your layout differs (ignored on Kaggle) ----
PROJECT_ROOT = os.path.abspath(".")                     # the folder that holds this notebook
DESKTOP_DATA = os.path.join(PROJECT_ROOT, "data")       # the datasets, read only (layout in the notes above)
DESKTOP_OUTPUTS = os.path.join(PROJECT_ROOT, "outputs") # landmarks, predictions, final weights
# Rebuilt every run: ~150k small image files plus MediaPipe, about 4 GB. Keep it on a fast local disk; under WSL
# use the Linux side (~/...), not a Windows folder under /mnt/c, which is much slower for many small files.
DESKTOP_SCRATCH = os.path.expanduser("~/fer_scratch")

INPUT_ROOT = "/kaggle/input" if ON_KAGGLE else DESKTOP_DATA
WORK_ROOT = "/kaggle/working" if ON_KAGGLE else DESKTOP_OUTPUTS
TEMP_ROOT = "/kaggle/temp" if ON_KAGGLE else DESKTOP_SCRATCH
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(TEMP_ROOT, exist_ok=True)


def find_input(*layouts):
    """
    The first dataset folder that exists under INPUT_ROOT among the given layouts (each a tuple of folder names),
    also searched one or more levels deeper (Kaggle mounts and unzipped downloads differ).
    """
    for parts in layouts:
        if os.path.isdir(os.path.join(INPUT_ROOT, *parts)):
            return os.path.join(INPUT_ROOT, *parts)
    for parts in layouts:
        matches = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", *parts), recursive=True))
        if matches:
            return matches[0]
    expected = " or ".join(os.path.join(INPUT_ROOT, *parts) for parts in layouts)
    raise FileNotFoundError(f"Dataset folder not found. Expected {expected}, with one subfolder per class inside.")


COMBINED_TRAIN = find_input(("combined", "combined", "train"), ("combined", "train"))
COMBINED_TEST = find_input(("combined", "combined", "test"), ("combined", "test"))
CROSS_TEST_SOURCE = find_input(("emotion-detection-fer", "test"))

PREPROCESSED_TRAIN = os.path.join(TEMP_ROOT, "preprocessed", "train")
PREPROCESSED_TEST = os.path.join(TEMP_ROOT, "preprocessed", "test")
SPLIT_TRAIN = os.path.join(TEMP_ROOT, "split", "train")
SPLIT_VAL = os.path.join(TEMP_ROOT, "split", "val")
OVERSAMPLED_TRAIN = os.path.join(TEMP_ROOT, "oversampled", "train")
CROSS_TEST = os.path.join(TEMP_ROOT, "cross", "test")

print(f"Arm: {VARIANT} | {'Kaggle' if ON_KAGGLE else 'desktop'}")
print(f"inputs: {INPUT_ROOT}\noutputs: {WORK_ROOT}\nscratch: {TEMP_ROOT}")
for name, path in (("training pool", COMBINED_TRAIN), ("test set", COMBINED_TEST), ("cross-dataset test", CROSS_TEST_SOURCE)):
    classes = sorted(d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)))
    n_files = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
    print(f"{name}: {path} -> {len(classes)} classes, {n_files:,} files ({', '.join(classes)})")

In [ ]:
import os
import shutil
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Apply CLAHE to improve contrast
def preprocess_image(image_path, target_size=(224, 224)):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, target_size)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

# Preprocess and save images to a new directory
def preprocess_dataset(input_dir, output_dir, target_size=(224, 224)):
    os.makedirs(output_dir, exist_ok=True)
    for emotion in os.listdir(input_dir):
        emotion_dir = os.path.join(input_dir, emotion)
        output_emotion_dir = os.path.join(output_dir, emotion)
        os.makedirs(output_emotion_dir, exist_ok=True)

        for img_name in os.listdir(emotion_dir):
            img_path = os.path.join(emotion_dir, img_name)
            processed_img = preprocess_image(img_path, target_size=target_size)
            output_path = os.path.join(output_emotion_dir, img_name)
            cv2.imwrite(output_path, processed_img)

# Preprocess combined training and testing datasets
preprocessed_train = PREPROCESSED_TRAIN
preprocessed_test = PREPROCESSED_TEST

combined_train = COMBINED_TRAIN
combined_test = COMBINED_TEST

preprocess_dataset(combined_train, preprocessed_train)
preprocess_dataset(combined_test, preprocessed_test)
print("Preprocessing completed!")

In [ ]:
import hashlib

# ==============================
#   Train / Validation Split
# ==============================
# Early stopping and the LR schedule monitor this held-out slice of the training data.
# The test set is only used once, after training.

def split_train_val(src_dir, train_out, val_out, val_frac=0.1, seed=42):
    """Stratified per-class split of src_dir into train_out and val_out (files are copied)."""
    rng = np.random.default_rng(seed)
    for out_dir in (train_out, val_out):
        shutil.rmtree(out_dir, ignore_errors=True)
    for class_name in sorted(os.listdir(src_dir)):
        class_dir = os.path.join(src_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        files = sorted(os.listdir(class_dir))
        rng.shuffle(files)
        n_val = max(1, int(round(len(files) * val_frac)))
        for out_dir, subset in ((val_out, files[:n_val]), (train_out, files[n_val:])):
            out_class_dir = os.path.join(out_dir, class_name)
            os.makedirs(out_class_dir, exist_ok=True)
            for f in subset:
                shutil.copy(os.path.join(class_dir, f), out_class_dir)
        print(f" Class '{class_name}': {len(files) - n_val} train / {n_val} val")

def file_hashes(directory):
    """md5 of every file under directory (recursively)."""
    hashes = set()
    for root, _, files in os.walk(directory):
        for f in files:
            with open(os.path.join(root, f), "rb") as fh:
                hashes.add(hashlib.md5(fh.read()).hexdigest())
    return hashes

def count_md5_overlap(dir_a, dir_b, name_a="A", name_b="B"):
    """Report how many byte-identical images appear in both directories (informational only)."""
    overlap = len(file_hashes(dir_a) & file_hashes(dir_b))
    print(f"Byte-identical images shared by {name_a} and {name_b}: {overlap}")
    return overlap

preprocessed_train = PREPROCESSED_TRAIN
preprocessed_test = PREPROCESSED_TEST
split_train = SPLIT_TRAIN
split_val = SPLIT_VAL

split_train_val(preprocessed_train, split_train, split_val, val_frac=0.1, seed=SEED)
count_md5_overlap(split_train, split_val, "train", "val")
count_md5_overlap(split_train, preprocessed_test, "train", "test")

In [ ]:
import os
import shutil
import cv2
import numpy as np
import math
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

# ==============================
#   1) Helper Functions
# ==============================

def get_class_distribution(directory):
    """
    Dynamically discover how many images each class folder contains.
    Returns a dict: { 'class_folder_name': count_of_images, ... }
    """
    distribution = {}
    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return distribution

    classes = [d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))]
    for class_name in classes:
        class_dir = os.path.join(directory, class_name)
        distribution[class_name] = len(os.listdir(class_dir))
    return distribution

def oversample_based_on_distribution(
    input_dir,
    output_dir,
    class_distribution,
    target_size=(224, 224),
    fill_mode='nearest'
):
    """
    Dynamically oversample classes to match the size of the largest class.

    Each augmented copy gets a unique name (aug_<k>_<original name>.png). datagen.flow(save_to_dir=...)
    only adds a random 0-9999 suffix, so augmented images silently overwrote each other.

    Args:
        input_dir (str): Directory of the preprocessed dataset (by class folders).
        output_dir (str): Where to save the oversampled dataset (recreated on every run).
        class_distribution (dict): e.g. {'0': 4693, '1': 1153, '2': 4378, '3': 11986, ...}
        target_size (tuple): Image size (height, width).
        fill_mode (str): Fill mode for augmentation.

    Returns:
        dict: Expected number of images per class in output_dir.
    """
    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.1,
        fill_mode=fill_mode
    )
    shutil.rmtree(output_dir, ignore_errors=True)
    os.makedirs(output_dir, exist_ok=True)

    # 1. Find the largest class
    max_count = max(class_distribution.values()) if class_distribution else 0
    print(f"Largest class count (pre-balancing): {max_count}")

    expected_counts = {}
    # 2. For each class, compute how many times to replicate
    for class_name, file_count in class_distribution.items():
        input_class_dir = os.path.join(input_dir, class_name)
        output_class_dir = os.path.join(output_dir, class_name)
        os.makedirs(output_class_dir, exist_ok=True)

        files = os.listdir(input_class_dir)

        # Always copy the original images
        for f in files:
            shutil.copy(os.path.join(input_class_dir, f), output_class_dir)
        expected_counts[class_name] = len(files)

        # If class has no images, skip
        if file_count == 0:
            print(f"Skipping class '{class_name}' - no images.")
            continue

        # ratio = how many times bigger the largest class is, compared to the current class
        ratio = max_count // file_count
        if ratio <= 1:
            # This class is already near or at the largest
            continue

        print(f"Class {class_name} has {file_count} images; oversampling ratio: {ratio}")

        # Generate 'ratio - 1' new images per original
        oversample_factor = ratio - 1

        for f in files:
            img = cv2.imread(os.path.join(input_class_dir, f), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, target_size).astype('float32')[..., np.newaxis]  # (H, W, 1)
            stem = os.path.splitext(f)[0]
            for k in range(oversample_factor):
                augmented = datagen.random_transform(img)
                out_path = os.path.join(output_class_dir, f"aug_{k}_{stem}.png")
                cv2.imwrite(out_path, np.clip(augmented, 0, 255).astype('uint8'))
            expected_counts[class_name] += oversample_factor

    return expected_counts

def compute_dynamic_class_weights(generator):
    """
    Computes class weights dynamically based on the class distribution in the dataset.
    Args:
        generator (DirectoryIterator): The generator to compute class weights from.
    Returns:
        dict: Dictionary of class weights { class_index: weight, ... }
    """
    class_counts = Counter(generator.classes)
    print("Class distribution in generator:", dict(class_counts))

    classes = np.unique(generator.classes)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=generator.classes
    )
    return dict(enumerate(class_weights))

def visualize_class_distribution_and_weights(generator, class_weights_dict, title_prefix=""):
    """
    Visualizes the class distribution and weights as plots and a table.
    Args:
        generator: The data generator used to compute class distribution.
        class_weights_dict: Dictionary of class weights.
        title_prefix: String to prepend to plot titles (e.g. "Before Balancing" / "After Balancing").
    """
    class_indices = generator.class_indices
    # Sort class names by index
    sorted_class_names = sorted(class_indices, key=class_indices.get)

    # Count the number of samples per class
    class_counts = Counter(generator.classes)

    # Build arrays for plot
    class_sizes = []
    weights = []
    for cname in sorted_class_names:
        idx = class_indices[cname]
        class_sizes.append(class_counts[idx])
        weights.append(class_weights_dict.get(idx, 1.0))

    df = pd.DataFrame({
        "Class Name": sorted_class_names,
        "Number of Images": class_sizes,
        "Class Weight": weights
    })

    # Display the table
    print(f"\n{title_prefix} Class Distribution and Weights:")
    print(df)

    # Plot the class sizes
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, class_sizes, color='skyblue')
    plt.title(f"{title_prefix} Class Sizes")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.show()

    # Plot the class weights
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, weights, color='orange')
    plt.title(f"{title_prefix} Class Weights")
    plt.xlabel("Class")
    plt.ylabel("Weight")
    plt.xticks(rotation=45)
    plt.show()


# Paths (only the training split is oversampled; validation and test stay untouched)
split_train = SPLIT_TRAIN
oversampled_train = OVERSAMPLED_TRAIN

# 1. Compute & Print Distribution BEFORE Balancing
pre_balancing_dist = get_class_distribution(split_train)
print("\nClass Distribution (Before Balancing):")
for k, v in pre_balancing_dist.items():
    print(f" Class '{k}': {v} images")

# 2. Oversample
expected_counts = oversample_based_on_distribution(
    input_dir=split_train,
    output_dir=oversampled_train,
    class_distribution=pre_balancing_dist,  # Computed distribution
    target_size=(224, 224),
    fill_mode='nearest'
)

# 3. Compute & Print Distribution AFTER Balancing
post_balancing_dist = get_class_distribution(oversampled_train)
print("\nClass Distribution (After Balancing):")
for k, v in post_balancing_dist.items():
    print(f" Class '{k}': {v} images")
assert post_balancing_dist == expected_counts, (
    f"Oversampled counts {post_balancing_dist} differ from expected {expected_counts}; files were overwritten"
)

# 4. Create Generators & Compute Weights
img_size = (224, 224)
batch_size = 16

train_datagen = ImageDataGenerator(rescale=1.0/255.0)
oversampled_generator = train_datagen.flow_from_directory(
    oversampled_train,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True
)

class_weights_dict = compute_dynamic_class_weights(oversampled_generator)
print("\nComputed Class Weights (After Balancing):")
print(class_weights_dict)

# 5. Visualize final distribution & weights
visualize_class_distribution_and_weights(oversampled_generator, class_weights_dict, title_prefix="After Balancing")


## Face geometry branch
For the geometry arm, every image gets 34 measurements of the mouth, brows, eyes and nose.
1. **Landmarks, computed once.** MediaPipe's Face Landmarker (the 478-point face mesh) runs on every
   preprocessed image. A 50% black border is added first, then 25%, then none, because the detector expects
   some background around a face. Results go to `landmarks/landmarks_<set>.csv.gz` with a `found` flag.
   Each image is also detected a second time as its mirror image. MediaPipe needs a newer numpy than
   TensorFlow, so it is installed into its own folder and runs as a separate process. A later session reuses
   the CSVs from an attached output.
2. **Measurements.** The landmarks are rotated so the eyes are level, centred between the eyes and divided by
   the distance between the eye centres, which makes them independent of image size and head tilt.
   Left/right pairs enter only as means and absolute differences, so a mirrored face gives the same numbers
   (training uses horizontal flips). The measurements of the two detections (image and mirror image) are
   averaged, which halves the detector's noise on these small faces. They are then standardised with training-split statistics,
   clipped to ±5, and a 35th input says whether a face was found (all zeros when not).
3. **Batches.** Every batch carries `{"image_input": images, "geometry_input": measurements}`. Augmented
   oversampled copies use their source image's measurements, and MixUp mixes both inputs with the same weight.

In [ ]:
%%writefile extract_landmarks.py
"""
Facial landmarks for the FER images with MediaPipe's Face Landmarker (478-point face mesh).

Runs in its own process with its own packages: MediaPipe needs a newer numpy than TensorFlow.
Each image is detected twice: as it is and mirrored. Writes one CSV row per image: key ("<class>/<file stem>"),
found / found_mirror (0/1), the border that worked, and the x/y position (pixels) of each requested landmark in
the image (x<id>, y<id>) and in the mirrored image (mx<id>, my<id>).
"""
import argparse
import csv
import gzip
import os
import sys
from multiprocessing import Pool

sys.modules["tensorflow"] = None  # MediaPipe imports TensorFlow only for its docs tooling; keep it out of this process

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp")
BORDERS = (0.5, 0.25, 0.0)  # black border around the face crop, tried in order: the detector expects some background

_landmarker = None
_ids = None


def init_worker(model_path, ids):
    global _landmarker, _ids
    options = vision.FaceLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=model_path),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.3,
        min_face_presence_confidence=0.3,
    )
    _landmarker = vision.FaceLandmarker.create_from_options(options)
    _ids = ids


def detect_once(gray):
    """(border that worked, landmark points) for one image, or (-1.0, None)."""
    for border in BORDERS:
        pad = int(round(border * max(gray.shape)))
        image = cv2.copyMakeBorder(gray, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0) if pad else gray
        rgb = np.ascontiguousarray(cv2.cvtColor(image, cv2.COLOR_GRAY2RGB))
        result = _landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        if result.face_landmarks:
            height, width = image.shape
            points = result.face_landmarks[0]
            return border, [(points[i].x * width - pad, points[i].y * height - pad) for i in _ids]
    return -1.0, None


def detect(job):
    key, path = job
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return key, -1.0, None, None
    border, points = detect_once(gray)
    _, mirror_points = detect_once(np.ascontiguousarray(gray[:, ::-1]))
    return key, border, points, mirror_points


def list_images(root):
    jobs = []
    for cls in sorted(os.listdir(root)):
        folder = os.path.join(root, cls)
        if os.path.isdir(folder):
            for name in sorted(os.listdir(folder)):
                if name.lower().endswith(IMAGE_EXTS):
                    jobs.append((f"{cls}/{os.path.splitext(name)[0]}", os.path.join(folder, name)))
    return jobs


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--images", required=True, help="folder with one subfolder per class")
    parser.add_argument("--out", required=True, help="output .csv.gz")
    parser.add_argument("--model", required=True, help="face_landmarker.task")
    parser.add_argument("--ids", required=True, help="comma-separated landmark ids to keep")
    parser.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    args = parser.parse_args()

    ids = [int(i) for i in args.ids.split(",")]
    jobs = list_images(args.images)
    coords = [f"{axis}{i}" for i in ids for axis in ("x", "y")]
    header = ["key", "found", "found_mirror", "border"] + coords + [f"m{c}" for c in coords]
    tmp_path = args.out + ".tmp"
    found = found_mirror = 0
    with Pool(args.workers, initializer=init_worker, initargs=(args.model, ids)) as pool, \
            gzip.open(tmp_path, "wt", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(header)
        for n, (key, border, points, mirror_points) in enumerate(pool.imap(detect, jobs, chunksize=32), start=1):
            found += points is not None
            found_mirror += mirror_points is not None
            values = lambda pts: [f"{v:.2f}" for xy in pts for v in xy] if pts is not None else [""] * (2 * len(ids))
            writer.writerow([key, int(points is not None), int(mirror_points is not None), border]
                            + values(points) + values(mirror_points))
            if n % 5000 == 0 or n == len(jobs):
                print(f"{n}/{len(jobs)} images, face found in {found / n:.1%} (mirror image: {found_mirror / n:.1%})",
                      flush=True)
    os.replace(tmp_path, args.out)


if __name__ == "__main__":
    main()

In [ ]:
# =================================
# Face geometry: landmarks (computed once) -> 34 measurements per image
# =================================
import re
import shutil
import subprocess
import sys
import urllib.request
from tensorflow.keras.utils import Sequence

LANDMARK_DIR = os.path.join(WORK_ROOT, "landmarks")            # kept in the output; later sessions reuse it
LANDMARK_PKG_DIR = os.path.join(TEMP_ROOT, "landmark_pkgs")    # MediaPipe + its numpy, apart from TensorFlow's
MEDIAPIPE_VERSION = "1.0.1"
LANDMARKER_URL = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
                  "face_landmarker/float16/1/face_landmarker.task")
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp")

# Face-mesh landmark ids used by the measurements
MOUTH_OUTER_UPPER = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
MOUTH_OUTER_LOWER = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
MOUTH_INNER_UPPER = [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
MOUTH_INNER_LOWER = [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
OTHER_POINTS = [1, 2, 10, 33, 70, 98, 105, 107, 133, 145, 152, 159, 168, 263, 300, 327, 334, 336, 362, 374, 386]
LANDMARK_IDS = sorted(set(MOUTH_OUTER_UPPER + MOUTH_OUTER_LOWER + MOUTH_INNER_UPPER + MOUTH_INNER_LOWER + OTHER_POINTS))

GEOMETRY_FEATURES = [
    # mouth (15)
    "mouth_width", "mouth_opening", "lip_height", "mouth_aspect", "corner_lift", "corner_asymmetry",
    "upper_lip_curve", "lower_lip_curve", "inner_upper_lip_curve", "inner_lower_lip_curve",
    "mouth_drop", "corner_height", "upper_lip_to_nose", "mouth_shift", "lip_to_chin",
    # brows (7)
    "inner_brow_raise", "outer_brow_raise", "mid_brow_raise", "brow_gap", "brow_slope", "brow_asymmetry",
    "brow_to_bridge",
    # eyes (6)
    "eye_opening", "eye_width", "eye_aspect", "eye_asymmetry", "upper_lid_raise", "lower_lid_raise",
    # nose (4)
    "nostril_width", "nose_length", "nose_tip_to_lip", "nose_wing_to_corner",
    # head pose and face shape (2)
    "head_turn", "face_height",
]
GEOMETRY_DIM = len(GEOMETRY_FEATURES) + 1        # + the "face found" flag
MISSING_GEOMETRY = np.zeros(GEOMETRY_DIM, dtype="float32")


def geometry_features(points):
    """
    34 measurements per face from landmark points of shape (N, len(LANDMARK_IDS), 2), in pixels.
    The points are rotated so the eyes are level, centred between the eyes and divided by the distance between
    the eye centres. Left/right pairs enter only as means and absolute differences, so a mirrored face gives the
    same numbers.
    """
    col = {lid: k for k, lid in enumerate(LANDMARK_IDS)}
    eye_a = (points[:, col[33]] + points[:, col[133]]) / 2
    eye_b = (points[:, col[362]] + points[:, col[263]]) / 2
    centre = (eye_a + eye_b) / 2
    eye_line = eye_b - eye_a
    eye_distance = np.maximum(np.linalg.norm(eye_line, axis=1), 1e-6)
    dx = np.where(np.abs(eye_line[:, 0]) < 1e-6, 1e-6, eye_line[:, 0])
    angle = np.arctan(eye_line[:, 1] / dx)        # tilt of the eye line, in (-90, 90) degrees
    cos, sin = np.cos(-angle)[:, None], np.sin(-angle)[:, None]
    rel = points - centre[:, None, :]
    q = np.stack([rel[..., 0] * cos - rel[..., 1] * sin,
                  rel[..., 0] * sin + rel[..., 1] * cos], axis=-1) / eye_distance[:, None, None]

    P = lambda lid: q[:, col[lid]]                # x to the right, y down, in eye distances
    x = lambda lid: P(lid)[:, 0]
    y = lambda lid: P(lid)[:, 1]
    dist = lambda a, b: np.linalg.norm(P(a) - P(b), axis=1)

    def curve(ids):
        """Curvature a of the best-fitting parabola y = a x^2 + b x + c through a lip contour."""
        xs = np.stack([x(i) for i in ids], axis=1)
        ys = np.stack([y(i) for i in ids], axis=1)
        X = np.stack([xs ** 2, xs, np.ones_like(xs)], axis=-1)
        XtX = np.einsum("nki,nkj->nij", X, X) + 1e-6 * np.eye(3)
        Xty = np.einsum("nki,nk->ni", X, ys)
        return np.linalg.solve(XtX, Xty[..., None])[:, 0, 0]

    lip_mid = (y(13) + y(14)) / 2
    lift_a, lift_b = lip_mid - y(61), lip_mid - y(291)            # > 0: lip corner above the lip centre
    inner_a, inner_b = y(133) - y(107), y(362) - y(336)           # inner brow above the inner eye corner
    open_a, open_b = dist(159, 145), dist(386, 374)
    lids_a, lids_b = (y(33) + y(133)) / 2, (y(362) + y(263)) / 2  # eye-corner line of each eye
    features = {
        "mouth_width": dist(61, 291),
        "mouth_opening": dist(13, 14),
        "lip_height": dist(0, 17),
        "mouth_aspect": dist(13, 14) / np.maximum(dist(61, 291), 1e-6),
        "corner_lift": (lift_a + lift_b) / 2,
        "corner_asymmetry": np.abs(lift_a - lift_b),
        "upper_lip_curve": curve(MOUTH_OUTER_UPPER),
        "lower_lip_curve": curve(MOUTH_OUTER_LOWER),
        "inner_upper_lip_curve": curve(MOUTH_INNER_UPPER),
        "inner_lower_lip_curve": curve(MOUTH_INNER_LOWER),
        "mouth_drop": lip_mid,
        "corner_height": (y(61) + y(291)) / 2,
        "upper_lip_to_nose": dist(2, 0),
        "mouth_shift": np.abs((x(13) + x(14)) / 2 - x(2)),
        "lip_to_chin": dist(17, 152),
        "inner_brow_raise": (inner_a + inner_b) / 2,
        "outer_brow_raise": ((y(33) - y(70)) + (y(263) - y(300))) / 2,
        "mid_brow_raise": ((y(159) - y(105)) + (y(386) - y(334))) / 2,
        "brow_gap": dist(107, 336),
        "brow_slope": ((y(70) - y(107)) + (y(300) - y(336))) / 2,   # > 0: inner ends higher (sad); < 0: lowered (angry)
        "brow_asymmetry": np.abs(inner_a - inner_b),
        "brow_to_bridge": y(168) - (y(107) + y(336)) / 2,
        "eye_opening": (open_a + open_b) / 2,
        "eye_width": (dist(33, 133) + dist(362, 263)) / 2,
        "eye_aspect": (open_a + open_b) / np.maximum(dist(33, 133) + dist(362, 263), 1e-6),
        "eye_asymmetry": np.abs(open_a - open_b),
        "upper_lid_raise": ((lids_a - y(159)) + (lids_b - y(386))) / 2,
        "lower_lid_raise": ((y(145) - lids_a) + (y(374) - lids_b)) / 2,
        "nostril_width": dist(98, 327),
        "nose_length": dist(168, 2),
        "nose_tip_to_lip": dist(1, 0),
        "nose_wing_to_corner": (dist(98, 61) + dist(327, 291)) / 2,
        "head_turn": np.abs(dist(33, 1) - dist(263, 1)),
        "face_height": dist(10, 152),
    }
    return np.stack([features[name] for name in GEOMETRY_FEATURES], axis=1).astype("float32")


def run_streaming(cmd, env=None):
    """Run a command and print its output as it arrives (MediaPipe's own log lines are dropped)."""
    noise = ("W0000", "I0000", "INFO: Created", "WARNING: All log messages", "WARNING: Logging before")
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if not line.startswith(noise):
            print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{' '.join(cmd[:3])} ... failed with exit code {proc.returncode}")


def ensure_landmark_tools():
    """MediaPipe in its own folder (not mixed into TensorFlow's packages) and the face-landmarker model."""
    if not os.path.isdir(os.path.join(LANDMARK_PKG_DIR, "mediapipe")):
        run_streaming([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                       "--target", LANDMARK_PKG_DIR, "--ignore-installed", f"mediapipe=={MEDIAPIPE_VERSION}"])
    model_path = os.path.join(LANDMARK_PKG_DIR, "face_landmarker.task")
    if not os.path.exists(model_path):
        urllib.request.urlretrieve(LANDMARKER_URL, model_path)
    return model_path


def count_images(root):
    return sum(f.lower().endswith(IMAGE_EXTS) for _, _, files in os.walk(root) for f in files)


def landmarks_for(name, image_root):
    """Landmarks of every image under image_root: from this session, an attached earlier output, or a new run."""
    out = os.path.join(LANDMARK_DIR, f"landmarks_{name}.csv.gz")
    if not os.path.exists(out):
        os.makedirs(LANDMARK_DIR, exist_ok=True)
        earlier = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", f"landmarks_{name}.csv.gz"), recursive=True))
        if earlier:
            shutil.copy(earlier[0], out)
            print(f"Reusing landmarks from {earlier[0]}")
        else:
            model_path = ensure_landmark_tools()
            env = dict(os.environ, PYTHONPATH=os.pathsep.join(filter(None, [LANDMARK_PKG_DIR, os.environ.get("PYTHONPATH")])))
            print(f"Detecting face landmarks in {image_root} ...")
            run_streaming([sys.executable, os.path.abspath("extract_landmarks.py"), "--images", image_root,
                           "--out", out, "--model", model_path, "--ids", ",".join(map(str, LANDMARK_IDS))], env=env)
    frame = pd.read_csv(out, dtype={"key": str})
    n_images = count_images(image_root)
    assert len(frame) == n_images, f"{out} has {len(frame)} rows but {image_root} has {n_images} images; delete it to recompute"
    return frame


def points_of(frame, rows, prefix=""):
    """Landmark points (N, len(LANDMARK_IDS), 2) of the selected rows; prefix "m" = the mirror-image detection."""
    return np.stack([frame.loc[rows, [f"{prefix}x{i}" for i in LANDMARK_IDS]].to_numpy(float),
                     frame.loc[rows, [f"{prefix}y{i}" for i in LANDMARK_IDS]].to_numpy(float)], axis=-1)


def raw_geometry(frame, with_views=False):
    """
    Unstandardised measurements and the found flag for every row of a landmark table. Each face was detected on
    the image and on its mirror image; the measurements of the two detections are averaged (or the one that
    worked is used), which lowers the detector's noise.
    """
    seen, seen_mirror = frame["found"].to_numpy() == 1, frame["found_mirror"].to_numpy() == 1
    view_a = np.zeros((len(frame), len(GEOMETRY_FEATURES)), dtype="float32")
    view_b = np.zeros_like(view_a)
    if seen.any():
        view_a[seen] = geometry_features(points_of(frame, seen))
    if seen_mirror.any():
        view_b[seen_mirror] = geometry_features(points_of(frame, seen_mirror, "m"))
    both = seen & seen_mirror
    features = np.where(seen[:, None], view_a, view_b)
    features[both] = (view_a[both] + view_b[both]) / 2
    found = seen | seen_mirror
    return (features, found, view_a, view_b, both) if with_views else (features, found)


GEOMETRY_TABLE = {}  # "<set>/<class>/<stem>" -> standardised measurements + found flag


def add_to_geometry_table(name, frame):
    """Standardise with the training-split statistics and add the rows of one landmark table."""
    features, found = raw_geometry(frame)
    z = np.clip((features - GEOMETRY_MEAN) / GEOMETRY_STD, -5, 5)
    z[~found] = 0
    vectors = np.concatenate([z, found[:, None].astype("float32")], axis=1)
    GEOMETRY_TABLE.update(zip((f"{name}/{key}" for key in frame["key"]), vectors))


def geometry_key(path, source):
    """Landmark-table key of an image file. Augmented copies (aug_<k>_<stem>.png) use their source image."""
    stem = re.sub(r"^aug_\d+_", "", os.path.splitext(os.path.basename(path))[0])
    return f"{source}/{os.path.basename(os.path.dirname(path))}/{stem}"


class GeometrySequence(Sequence):
    """
    Wraps a Keras directory iterator so every batch is ({"image_input": images, "geometry_input": vectors}, labels).
    `source` names the landmark table: "train" (also for the validation split), "test" or "cross".
    """
    def __init__(self, iterator, source, **kwargs):
        super().__init__(**kwargs)
        self.iterator = iterator
        self.source = source
        self.batch_size = iterator.batch_size
        keys = [geometry_key(p, source) for p in iterator.filepaths]
        missing = sum(k not in GEOMETRY_TABLE for k in keys)
        if missing:
            print(f"Warning: {missing} of {len(keys)} images have no landmark row; they get the 'no face' vector")
        self.features = np.stack([GEOMETRY_TABLE.get(k, MISSING_GEOMETRY) for k in keys]).astype("float32")
        self.face_found = self.features[:, -1].copy()

    def __len__(self):
        return len(self.iterator)

    def __getitem__(self, idx):
        images, labels = self.iterator[idx]
        index_array = self.iterator.index_array[idx * self.batch_size:(idx + 1) * self.batch_size]
        return {"image_input": images, "geometry_input": self.features[index_array]}, labels

    def on_epoch_end(self):
        self.iterator.on_epoch_end()

    def __getattr__(self, name):
        # classes, class_indices, filenames, directory, reset(), ... come from the wrapped iterator
        if name == "iterator":
            raise AttributeError(name)
        return getattr(self.iterator, name)


if USE_GEOMETRY:
    landmark_frames = {"train": landmarks_for("train", PREPROCESSED_TRAIN), "test": landmarks_for("test", PREPROCESSED_TEST)}

    # Standardisation statistics from the training split only (faces found)
    train_features, train_found, view_a, view_b, both_views = raw_geometry(landmark_frames["train"], with_views=True)
    split_keys = {f"{cls}/{os.path.splitext(f)[0]}" for cls in os.listdir(SPLIT_TRAIN)
                  for f in os.listdir(os.path.join(SPLIT_TRAIN, cls))}
    use = train_found & landmark_frames["train"]["key"].isin(split_keys).to_numpy()
    assert use.sum() >= 2, "Too few faces found in the training split to standardise the measurements"
    GEOMETRY_MEAN = train_features[use].mean(axis=0)
    GEOMETRY_STD = train_features[use].std(axis=0) + 1e-6
    for name, frame in landmark_frames.items():
        add_to_geometry_table(name, frame)

    # Agreement between the two detections of the same face (image vs mirror image): low = mostly detector noise
    agreement = pd.Series([np.corrcoef(view_a[both_views, j], view_b[both_views, j])[0, 1]
                           for j in range(len(GEOMETRY_FEATURES))], index=GEOMETRY_FEATURES)
    print("Agreement between the two detections (correlation over training faces), least reliable first:")
    print(agreement.sort_values().head(8).round(2).to_string(), "\n")

    # How often a face was found, per class and set
    coverage = pd.DataFrame({name: frame.groupby(frame["key"].str.split("/").str[0])["found"].mean()
                             for name, frame in landmark_frames.items()})
    print("Face found (% of images):")
    print((100 * coverage).round(1))

    # Sanity check: typical measurements per class (training split, faces found, in standard deviations)
    by_class = pd.DataFrame(np.clip((train_features[use] - GEOMETRY_MEAN) / GEOMETRY_STD, -5, 5), columns=GEOMETRY_FEATURES)
    by_class["class"] = landmark_frames["train"]["key"].str.split("/").str[0].to_numpy()[use]
    print("\nMean standardised measurement per class (training split):")
    print(by_class.groupby("class")[["corner_lift", "mouth_opening", "inner_brow_raise", "brow_gap",
                                     "eye_opening", "upper_lip_to_nose"]].mean().round(2))

In [ ]:
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout,
    BatchNormalization, Conv2D, Multiply,
    Lambda, Concatenate, Reshape, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize

# ---- NEW IMPORTS FOR ViT ----
from transformers import TFViTModel

In [ ]:

class ViTWrapper(tf.keras.layers.Layer):
    """
    A custom Keras layer that wraps TFViTModel and ensures compatibility with the functional API.
    """
    def __init__(self, pretrained_name="google/vit-base-patch16-224-in21k", **kwargs):
        super().__init__(**kwargs)
        self.pretrained_name = pretrained_name
        self.vit_model = TFViTModel.from_pretrained(pretrained_name)

    def call(self, inputs, training=False):
        # Transpose to channels-first for ViT
        x = tf.transpose(inputs, perm=[0, 3, 1, 2])  # (B, 3, 224, 224)
        # Normalize inputs: the in21k checkpoint was trained with mean = std = 0.5
        x = (x - 0.5) / 0.5
        outputs = self.vit_model(pixel_values=x, training=training)
        return outputs.pooler_output

    def freeze(self):
        """Freeze the whole ViT. The model-level flag overrides every sub-layer."""
        self.vit_model.trainable = False

    def unfreeze_layers(self):
        """Unfreeze the whole ViT."""
        self.vit_model.trainable = True

    def get_config(self):
        config = super().get_config()
        config.update({"pretrained_name": self.pretrained_name})
        return config


In [ ]:
# -----------------------------
# Squash Activation Function for Capsules
# -----------------------------
def squash(s, axis=-1, epsilon=1e-7):
    s_squared_norm = tf.reduce_sum(tf.square(s), axis=axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm)
    return scale * s / tf.sqrt(s_squared_norm + epsilon)

# -----------------------------
# Advanced Capsule Fusion Block (CFB)
# -----------------------------
class AdvancedCapsuleFusionBlock(tf.keras.layers.Layer):
    """
    Advanced Capsule Fusion Block that fuses several feature vectors (one capsule per branch) using an iterative
    dynamic routing algorithm with enhanced transformations, residual updates, dropout, and batch normalization.
    With num_inputs=2 it is the original two-branch block (same layers, same names).
    """
    def __init__(self, capsule_dim=256, num_iterations=2, dropout_rate=0.4, num_inputs=2, **kwargs):
        super(AdvancedCapsuleFusionBlock, self).__init__(**kwargs)
        self.capsule_dim = capsule_dim
        self.num_iterations = num_iterations
        self.dropout_rate = dropout_rate
        self.num_inputs = num_inputs

        # Two-layer MLP + batch norm for each branch's transformation: mlp1/bn1, mlp2/bn2, ...
        for i in range(1, num_inputs + 1):
            setattr(self, f"mlp{i}", tf.keras.Sequential([
                Dense(capsule_dim, activation="relu"),
                Dropout(dropout_rate),
                Dense(capsule_dim, activation=None)
            ]))
            setattr(self, f"bn{i}", BatchNormalization())

    def call(self, inputs, training=False):
        # inputs: list [f1, f2, ...], one feature vector per branch
        assert len(inputs) == self.num_inputs, f"expected {self.num_inputs} inputs, got {len(inputs)}"
        # Transform each branch using its two-layer MLP: each (batch, capsule_dim)
        capsules = [getattr(self, f"bn{i}")(getattr(self, f"mlp{i}")(f), training=training)
                    for i, f in enumerate(inputs, start=1)]

        # Stack to form capsules: shape (batch, num_inputs, capsule_dim)
        u = tf.stack(capsules, axis=1)

        # Initialize routing logits b (batch, num_inputs) with zeros
        b = tf.zeros(shape=(tf.shape(u)[0], self.num_inputs), dtype=tf.float32)

        # Iterative dynamic routing
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=1)  # coupling coefficients (batch, num_inputs)
            c_expanded = tf.expand_dims(c, axis=-1)  # (batch, num_inputs, 1)
            # Weighted sum of capsule predictions
            s = tf.reduce_sum(c_expanded * u, axis=1)  # (batch, capsule_dim)
            v = squash(s, axis=-1)  # (batch, capsule_dim)

            # Residual update: encourage consistency with previous capsule output
            if i > 0:
                v = 0.5 * v + 0.5 * v_prev
            v_prev = v  # store for residual update

            v_expanded = tf.expand_dims(v, axis=1)  # (batch, 1, capsule_dim)
            # Update routing logits based on agreement
            b += tf.reduce_sum(u * v_expanded, axis=-1)  # (batch, num_inputs)
        return v  # Final fused capsule (batch, capsule_dim)


In [ ]:
# -----------------------------
# Positional Embedding Layer for TRB
# -----------------------------
class PositionalEmbedding(tf.keras.layers.Layer):
    """
    Learns a positional embedding for a fixed number of tokens.
    """
    def __init__(self, num_tokens, token_dim, **kwargs):
        super(PositionalEmbedding, self).__init__(**kwargs)
        self.num_tokens = num_tokens
        self.token_dim = token_dim

    def build(self, input_shape):
        # Create a learnable positional embedding matrix
        self.pos_embedding = self.add_weight(
            shape=(1, self.num_tokens, self.token_dim),
            initializer="uniform",
            trainable=True,
            name="pos_embedding"
        )
        super(PositionalEmbedding, self).build(input_shape)

    def call(self, inputs):
        # inputs shape: (batch, num_tokens, token_dim)
        return inputs + self.pos_embedding

# -----------------------------
# Advanced Transformer Encoder Block for TRB
# -----------------------------
class TransformerEncoderBlock(tf.keras.layers.Layer):
    """
    A full transformer encoder block with multi-head self-attention, feed-forward network,
    residual connections, and layer normalization.
    """
    def __init__(self, token_dim, num_heads, ff_dim, dropout_rate=0.4, **kwargs):
        super(TransformerEncoderBlock, self).__init__(**kwargs)
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=token_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(token_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
    
    def call(self, inputs, training=False):
        attn_output = self.mha(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# -----------------------------
# Advanced Transformer Refinement Block (TRB)
# -----------------------------
class AdvancedTransformerRefinementBlock(tf.keras.layers.Layer):
    """
    Refines the fused capsule features using a stack of transformer encoder blocks.
    The fused vector is projected into multiple tokens, positional embeddings are added,
    processed by transformer layers, and then aggregated.
    """
    def __init__(self, num_tokens=4, token_dim=64, num_heads=4, ff_dim=128, num_layers=2, dropout_rate=0.3, **kwargs):
        super(AdvancedTransformerRefinementBlock, self).__init__(**kwargs)
        self.num_tokens = num_tokens
        self.token_dim = token_dim
        self.proj = Dense(num_tokens * token_dim)
        self.reshape_layer = Lambda(lambda x: tf.reshape(x, (-1, num_tokens, token_dim)))
        self.pos_emb = PositionalEmbedding(num_tokens, token_dim)
        self.transformer_layers = [TransformerEncoderBlock(token_dim, num_heads, ff_dim, dropout_rate) for _ in range(num_layers)]
    
    def call(self, inputs, training=False):
        # inputs: fused capsule features (batch, capsule_dim)
        tokens = self.proj(inputs)                # (batch, num_tokens * token_dim)
        tokens = self.reshape_layer(tokens)         # (batch, num_tokens, token_dim)
        tokens = self.pos_emb(tokens)               # Add positional embedding
        for layer in self.transformer_layers:
            tokens = layer(tokens, training=training)
        # Aggregate tokens via mean pooling
        refined = tf.reduce_mean(tokens, axis=1)    # (batch, token_dim)
        return refined

# focal loss

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class MixupSequence(Sequence):
    def __init__(self, datagen, alpha=0.1, **kwargs):
        """
        Wraps a Keras ImageDataGenerator (Sequence) to perform MixUp on each batch.

        Args:
            datagen: A Keras Sequence (e.g., from flow_from_directory, or a GeometrySequence with dict inputs).
            alpha: Beta distribution parameter for mixing. Higher alpha means stronger mixing.
        """
        super().__init__(**kwargs)
        self.datagen = datagen
        self.alpha = alpha
        self.batch_size = datagen.batch_size
        self.steps = len(datagen)

    def __len__(self):
        # Return the number of batches per epoch
        return self.steps

    def __getitem__(self, idx):
        # Get the idx-th batch from the underlying generator
        batch_x, batch_y = self.datagen[idx]
        batch_size = batch_y.shape[0]

        # Sample lambda from Beta(alpha, alpha)
        lam = np.random.beta(self.alpha, self.alpha)

        # Shuffle batch indices
        index = np.random.permutation(batch_size)

        # Perform MixUp on the images (and on every other model input, with the same lambda and pairs) and labels
        mix = lambda a: lam * a + (1 - lam) * a[index, :]
        mixed_x = {name: mix(x) for name, x in batch_x.items()} if isinstance(batch_x, dict) else mix(batch_x)
        mixed_y = mix(batch_y)

        return mixed_x, mixed_y

    def on_epoch_end(self):
        # Let the wrapped generator reshuffle; otherwise every epoch repeats the first epoch's order
        self.datagen.on_epoch_end()


In [ ]:
def focal_loss(alpha, gamma=2.0):
    """
    Focal Loss for multi-class classification.
    
    Args:
        alpha: A list or array of weights for each class (class importance).
        gamma: Focusing parameter to reduce the loss contribution from easy examples.
    
    Returns:
        A loss function that can be used in model.compile().
    """
    def loss_fn(y_true, y_pred):
        # Ensure predictions are probabilities
        y_pred = tf.keras.backend.clip(y_pred, 1e-3, 1 - 1e-3)
        
        # Compute the cross-entropy loss
        ce_loss = -y_true * tf.math.log(y_pred)
        
        # Apply the focal loss adjustment
        focal_loss = alpha * tf.pow(1 - y_pred, gamma) * ce_loss
        
        # Return the mean focal loss over the batch
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))
    
    return loss_fn


In [ ]:


# =================================
# 1) Paths and Data Generators
# =================================

oversampled_train = OVERSAMPLED_TRAIN
split_val = SPLIT_VAL
# The test set is used untouched: no oversampling, no augmentation, and only after training
preprocessed_test = PREPROCESSED_TEST

img_size = (224, 224)
batch_size = 16

# Train Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,       # Increased rotation range
    width_shift_range=0.1,   # Increased shift
    height_shift_range=0.1,
    brightness_range=[0.8, 1.0],  # Vary brightness
    shear_range=0.1,              # Shear transformations
    zoom_range=0.15,               # Increased zoom range
    horizontal_flip=True,
    fill_mode='reflect'
)

# Validation/Test: rescale only
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create Generators
train_generator = train_datagen.flow_from_directory(
    oversampled_train,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)
val_generator = test_datagen.flow_from_directory(
    split_val,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    preprocessed_test,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
assert train_generator.class_indices == val_generator.class_indices == test_generator.class_indices

if USE_GEOMETRY:
    # Every batch also carries the face-geometry vector of each image (see the face geometry section)
    train_generator = GeometrySequence(train_generator, source="train")
    val_generator = GeometrySequence(val_generator, source="train")  # the validation split comes from the training images
    test_generator = GeometrySequence(test_generator, source="test")

# =================================
# 2) Class Weight Computation
# =================================

def compute_dynamic_class_weights(generator):
    class_counts = Counter(generator.classes)
    print("Class distribution in generator:", dict(class_counts))
    classes = np.unique(generator.classes)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=generator.classes
    )
    return dict(enumerate(class_weights))

def visualize_class_distribution_and_weights(generator, class_weights_dict, title_prefix=""):
    class_indices = generator.class_indices
    sorted_class_names = sorted(class_indices, key=class_indices.get)

    # Gather data
    class_counts = Counter(generator.classes)
    class_sizes = []
    weights = []
    for cname in sorted_class_names:
        idx = class_indices[cname]
        class_sizes.append(class_counts[idx])
        weights.append(class_weights_dict.get(idx, 1.0))

    # Create a DataFrame for printing
    df = pd.DataFrame({
        "Class Name": sorted_class_names,
        "Number of Images": class_sizes,
        "Class Weight": weights
    })
    print(f"\n{title_prefix} Class Distribution and Weights:")
    print(df)

    # Plot Class Sizes
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, class_sizes, color='skyblue')
    plt.title(f"{title_prefix} Class Sizes")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.show()

    # Plot Class Weights
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, weights, color='orange')
    plt.title(f"{title_prefix} Class Weights")
    plt.xlabel("Class")
    plt.ylabel("Weight")
    plt.xticks(rotation=45)
    plt.show()

class_weights_dict = compute_dynamic_class_weights(train_generator)
visualize_class_distribution_and_weights(train_generator, class_weights_dict, title_prefix="Oversampled Train")

# =================================
# 3) CBAM + VGG16 + ViT (Multi-Branch) Model Definition
# =================================

def attention_block(input_tensor):
    # Channel Attention
    avg_pool = GlobalAveragePooling2D()(input_tensor)  # (batch, channels)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=[1, 2], keepdims=False))(input_tensor)  # (batch, channels)

    reduced_channels = input_tensor.shape[-1] // 8  # Reduce dimensionality
    avg_dense = Dense(reduced_channels, activation="relu")(avg_pool)
    max_dense = Dense(reduced_channels, activation="relu")(max_pool)
    channel_attention = Dense(input_tensor.shape[-1], activation="sigmoid")(avg_dense + max_dense)
    channel_refined = Multiply()([input_tensor, Reshape((1, 1, -1))(channel_attention)])

    # Spatial Attention
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(channel_refined)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(channel_refined)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    spatial_attention = Conv2D(1, (7, 7), activation="sigmoid", padding="same")(concat)
    refined_output = Multiply()([channel_refined, spatial_attention])
    return refined_output


class VGGPreprocess(tf.keras.layers.Layer):
    """
    Converts the [0, 1] RGB model input into the caffe preprocessing VGG16's ImageNet weights
    were trained with: 0-255 range, BGR channel order, ImageNet mean subtracted.
    """
    def call(self, inputs):
        x = inputs * 255.0
        x = x[..., ::-1]  # RGB -> BGR
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)


def build_vgg16_vit_model(num_classes, num_geometry_features=0):
    """
    Creates a multi-branch model that fuses features from:
      - VGG16 + CBAM CNN branch
      - Vision Transformer (ViT) branch
      - face-geometry branch (only when num_geometry_features > 0): a small MLP on the landmark measurements
    Fusion is performed using an advanced Capsule Fusion Block followed by an advanced Transformer Refinement Block.
    """
    input_tensor = Input(shape=(224, 224, 3), name="image_input")

    # VGG16 + CBAM Branch
    base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers[:-8]:
        layer.trainable = False
    vgg_input = VGGPreprocess(name="vgg_preprocess")(input_tensor)
    vgg_features = base_model(vgg_input)
    vgg_features = attention_block(vgg_features)
    vgg_features = GlobalAveragePooling2D()(vgg_features)
    vgg_features = Dense(256, activation="relu")(vgg_features)

    # ViT Branch (normalizes its own input)
    vit_branch = ViTWrapper(pretrained_name="google/vit-base-patch16-224-in21k", name="vit_wrapper")
    vit_features = vit_branch(input_tensor)

    branches, inputs = [vgg_features, vit_features], [input_tensor]

    # Face-geometry branch: 34 landmark measurements + a found flag -> 256-d, the third capsule
    if num_geometry_features:
        geometry_input = Input(shape=(num_geometry_features,), name="geometry_input")
        geometry_features = Dense(128, activation="relu", name="geometry_dense_1")(geometry_input)
        geometry_features = Dropout(0.2, name="geometry_dropout")(geometry_features)
        geometry_features = Dense(256, activation="relu", name="geometry_dense_2")(geometry_features)
        branches.append(geometry_features)
        inputs.append(geometry_input)

    # Fuse features using the Advanced Capsule Fusion Block (one capsule per branch)
    fused_capsule = AdvancedCapsuleFusionBlock(capsule_dim=256, num_iterations=2, dropout_rate=0.4, num_inputs=len(branches))(branches)

    # Refine fused features with the Advanced Transformer Refinement Block
    refined_features = AdvancedTransformerRefinementBlock(num_tokens=4, token_dim=64, num_heads=4, ff_dim=128, num_layers=2, dropout_rate=0.4)(fused_capsule)

    # Final Classification Layer
    final_output = Dense(num_classes, activation="softmax")(refined_features)

    model = Model(inputs=inputs if len(inputs) > 1 else input_tensor, outputs=final_output, name="Advanced_VGG16_CBAM_ViT_Model")
    return model


def count_vit_trainable_weights(model):
    """Number of ViT weight tensors the optimizer will update (0 means the ViT is frozen)."""
    n_vit = len(model.get_layer("vit_wrapper").trainable_weights)
    print(f"Trainable weight tensors: ViT {n_vit}, whole model {len(model.trainable_weights)}")
    return n_vit

# =================================
# 4) Training Utilities
# =================================

def evaluate_model(model, test_generator, history):
    # Evaluate on test data
    test_loss, test_accuracy = model.evaluate(test_generator)
    print(f"\nFinal Test Loss: {test_loss:.4f}")
    print(f"Final Test Accuracy: {test_accuracy:.4f}")

    # Accuracy & Loss Plots (skipped when the weights were loaded, not trained)
    if history is not None:
        plt.figure(figsize=(12, 5))

        # Accuracy
        plt.subplot(1, 2, 1)
        plt.plot(history['accuracy'], label='Train Accuracy')
        plt.plot(history['val_accuracy'], label='Validation Accuracy')
        plt.title('Model Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.legend()

        # Loss
        plt.subplot(1, 2, 2)
        plt.plot(history['loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Validation Loss')
        plt.title('Model Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()

        plt.show()

    # Predict once; the probabilities feed the report, confusion matrix and ROC
    test_generator.reset()
    y_pred_prob = model.predict(test_generator)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = test_generator.classes
    class_labels = list(test_generator.class_indices.keys())

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # ROC AUC
    y_true_binarized = label_binarize(y_true, classes=range(len(class_labels)))
    roc_auc = roc_auc_score(y_true_binarized, y_pred_prob, average="macro", multi_class="ovr")
    print(f"Macro-average ROC AUC: {roc_auc:.4f}")

    # Plot ROC Curve
    plt.figure(figsize=(10, 7))
    for i in range(len(class_labels)):
        fpr, tpr, _ = roc_curve(y_true_binarized[:, i], y_pred_prob[:, i])
        per_class_auc = roc_auc_score(y_true_binarized[:, i], y_pred_prob[:, i])
        plt.plot(fpr, tpr, label=f"{class_labels[i]} (AUC = {per_class_auc:.2f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
    plt.title('ROC Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.grid()
    plt.show()
    return y_pred_prob


def save_predictions(part, generator, probs):
    """Per-image probabilities for the ablation (one file per arm and test set)."""
    path = os.path.join(WORK_ROOT, f"predictions_{VARIANT}_{part}.npz")
    face_found = generator.face_found if isinstance(generator, GeometrySequence) else np.array([])
    class_names = sorted(generator.class_indices, key=generator.class_indices.get)
    np.savez(path, probs=probs, labels=np.asarray(generator.classes), files=np.asarray(generator.filenames),
             class_names=np.asarray(class_names), face_found=face_found)
    print(f"Saved {len(probs)} predictions to {path}")

# Normalize class weights for Focal Loss
alpha = np.array(list(class_weights_dict.values()))
alpha /= np.sum(alpha)

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define the modular loss function factory
def get_optimized_loss(alpha_weights, gamma_value=2.0):
    """
    Returns a compiled Focal Loss function.
    Setting gamma >= 2.0 aggressively suppresses gradients from easy majority classes (like Happy/Neutral).
    """
    return focal_loss(alpha=alpha_weights, gamma=gamma_value)

# 2. TTA Function: accepts an in-memory model or a saved-weights path
def evaluate_with_tta(model, test_generator, tta_steps=5):
    print("\n==========================================")
    print("Starting Test-Time Augmentation (TTA)...")
    print("==========================================")

    # If a weights path is passed instead of a model object, rebuild the model and load the weights
    if isinstance(model, str):
        print(f"Loading weights from disk: {model}")
        weights_path = model
        model = build_vgg16_vit_model(len(test_generator.class_indices), GEOMETRY_DIM if USE_GEOMETRY else 0)
        model.load_weights(weights_path).expect_partial()

    # Reset generator to start from the first batch
    test_generator.reset()
    y_true = test_generator.classes
    num_samples = len(y_true)
    num_classes = len(test_generator.class_indices)

    # Accumulate probability predictions across passes
    tta_predictions = np.zeros((num_samples, num_classes))

    # Step 1: Base unaugmented predictions
    print(f"Pass 1/{tta_steps}: Running base prediction pass...")
    test_generator.reset()
    tta_predictions += model.predict(test_generator, verbose=1)

    # Step 2: Configure light TTA transformations (flips and tiny zooms)
    tta_datagen = ImageDataGenerator(
        rescale=1.0/255.0,
        horizontal_flip=True,
        zoom_range=0.08
    )

    tta_generator = tta_datagen.flow_from_directory(
        test_generator.directory,
        target_size=test_generator.target_size,
        batch_size=test_generator.batch_size,
        class_mode="categorical",
        shuffle=False
    )

    if isinstance(test_generator, GeometrySequence):
        tta_generator = GeometrySequence(tta_generator, source=test_generator.source)

    # Step 3: Run augmented passes
    for step in range(1, tta_steps):
        print(f"Pass {step + 1}/{tta_steps}: Running augmented prediction pass...")
        tta_generator.reset()
        tta_predictions += model.predict(tta_generator, verbose=1)

    # Average the softmax probabilities across all passes
    tta_predictions /= tta_steps
    final_preds = np.argmax(tta_predictions, axis=1)

    acc = accuracy_score(y_true, final_preds)
    print("\n---------------------------------------------------------")
    print(f" FINAL TTA BOOSTED TEST ACCURACY: {acc:.4f} ")
    print("---------------------------------------------------------")
    print(classification_report(y_true, final_preds, target_names=list(test_generator.class_indices.keys())))
    return acc


In [ ]:
# =================================
# 5) Two-phase training
# =================================

# Parallel batch loading: ImageDataGenerator augmentation runs on the CPU
FIT_KWARGS = dict(workers=4, max_queue_size=20)

PHASES = [
    # Phase 1: ViT frozen; VGG16 blocks 4-5, CBAM, fusion blocks and head train
    {"phase": 1, "learning_rate": 3e-5, "epochs": 25, "vit_trainable": False, "unfreeze_all": False},
    # Phase 2: unfreeze everything, including VGG16 blocks 1-3 and the ViT
    {"phase": 2, "learning_rate": 1e-5, "epochs": 35, "vit_trainable": True, "unfreeze_all": True},
]
loss_fn = get_optimized_loss(alpha_weights=alpha, gamma_value=2.0)

def configure_phase(model, phase_config):
    """Set the phase's trainable layers and compile with a fresh AdamW, as in the original two-phase schedule."""
    if phase_config["unfreeze_all"]:
        for layer in model.layers:
            layer.trainable = True
    vit = model.get_layer("vit_wrapper")
    if phase_config["vit_trainable"]:
        vit.unfreeze_layers()
    else:
        vit.freeze()
    n_vit = count_vit_trainable_weights(model)
    assert (n_vit > 0) == phase_config["vit_trainable"], "ViT trainable state is wrong; check the tf.keras version cell"
    model.compile(
        optimizer=AdamW(learning_rate=phase_config["learning_rate"], weight_decay=1e-3),
        loss=loss_fn,
        metrics=["accuracy"]
    )

def train_model():
    """Runs the two-phase schedule. Returns the trained model and the history of both phases."""
    model = build_vgg16_vit_model(train_generator.num_classes, GEOMETRY_DIM if USE_GEOMETRY else 0)
    mixup_train_sequence = MixupSequence(train_generator, alpha=0.1)
    history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
    for phase_config in PHASES:
        configure_phase(model, phase_config)
        print(f"\n=== Phase {phase_config['phase']} (up to {phase_config['epochs']} epochs) ===")
        result = model.fit(
            mixup_train_sequence,
            validation_data=val_generator,
            epochs=phase_config["epochs"],
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
            ],
            **FIT_KWARGS
        )
        for key in history:
            history[key] += result.history[key]
    return model, history

def load_trained_model(weights_path):
    """The same model with the weights an earlier run of this arm saved, compiled for evaluation; no training."""
    assert os.path.exists(weights_path + ".index"), f"{weights_path}.index not found (give the path without .index)"
    model = build_vgg16_vit_model(train_generator.num_classes, GEOMETRY_DIM if USE_GEOMETRY else 0)
    configure_phase(model, PHASES[-1])
    model.load_weights(weights_path).expect_partial()
    return model

# =================================
# 6) Run Training & Evaluation
# =================================

if __name__ == "__main__":
    if PRETRAINED_WEIGHTS:
        print(f"Loading trained weights from {PRETRAINED_WEIGHTS}; training is skipped")
        final_model, training_history = load_trained_model(PRETRAINED_WEIGHTS), None
    else:
        final_model, training_history = train_model()

        # Save the final weights (TensorFlow checkpoint format); PRETRAINED_WEIGHTS can load them later
        save_path = os.path.join(WORK_ROOT, f"vgg16_cbam_vit_{VARIANT}")
        final_model.save_weights(save_path)
        print(f"\nWeights saved to {save_path}")

    # Standard Evaluation (first use of the test set); per-image predictions are kept for the ablation
    test_probs = evaluate_model(final_model, test_generator, training_history)
    save_predictions("test", test_generator, test_probs)

    # Test-Time Augmentation on the in-memory model
    evaluate_with_tta(final_model, test_generator, tta_steps=5)

# cross validate

In [ ]:
import os
import shutil
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Apply CLAHE to improve contrast
def preprocess_image(image_path, target_size=(224, 224)):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, target_size)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

# Preprocess and save images to a new directory
def preprocess_dataset(input_dir, output_dir, target_size=(224, 224)):
    os.makedirs(output_dir, exist_ok=True)
    for emotion in os.listdir(input_dir):
        emotion_dir = os.path.join(input_dir, emotion)
        output_emotion_dir = os.path.join(output_dir, emotion)
        os.makedirs(output_emotion_dir, exist_ok=True)

        for img_name in os.listdir(emotion_dir):
            img_path = os.path.join(emotion_dir, img_name)
            processed_img = preprocess_image(img_path, target_size=target_size)
            output_path = os.path.join(output_emotion_dir, img_name)
            cv2.imwrite(output_path, processed_img)

# Preprocess testing datasets
cross_test = CROSS_TEST

combined_test = CROSS_TEST_SOURCE

preprocess_dataset(combined_test, cross_test)
print("Preprocessing completed!")

In [ ]:
# Cross-Validation on a New Dataset (FERPlus as an example)
new_dataset_path = CROSS_TEST

# Create Test Generator
new_test_generator = test_datagen.flow_from_directory(
    new_dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

# Folders are matched to the training classes by sorted position ('disgust' vs 'disgusted', etc.)
train_class_names = sorted(train_generator.class_indices, key=train_generator.class_indices.get)
new_class_names = sorted(new_test_generator.class_indices, key=new_test_generator.class_indices.get)
assert len(train_class_names) == len(new_class_names), (train_class_names, new_class_names)
for train_name, new_name in zip(train_class_names, new_class_names):
    assert new_name.startswith(train_name[:4]), f"Class mismatch: {train_name} vs {new_name}"

# If the combined training data already contains these images, this is not a true cross-dataset test
count_md5_overlap(PREPROCESSED_TRAIN, new_dataset_path, "combined train", "emotion-detection-fer test")

if USE_GEOMETRY:
    add_to_geometry_table("cross", landmarks_for("cross", new_dataset_path))
    new_test_generator = GeometrySequence(new_test_generator, source="cross")

# Evaluate on the New Dataset
def evaluate_on_new_dataset(model, test_generator, dataset_name="New Dataset"):
    """
    Evaluate the trained model on another dataset.
    """
    test_loss, test_accuracy = model.evaluate(test_generator)
    print(f"\n{dataset_name} - Test Loss: {test_loss:.4f}")
    print(f"{dataset_name} - Test Accuracy: {test_accuracy:.4f}")

    # Classification Report
    probs = model.predict(test_generator)
    y_pred = np.argmax(probs, axis=1)
    y_true = test_generator.classes
    class_labels = list(test_generator.class_indices.keys())

    print(f"\n{dataset_name} - Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f"{dataset_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()
    return probs

# Evaluate on FERPlus (or any other dataset)
cross_probs = evaluate_on_new_dataset(final_model, new_test_generator, dataset_name="emotion-detection-fer")
save_predictions("cross", new_test_generator, cross_probs)


## Ablation: image only vs image + face geometry
Compares the two arms on the same test images: per-class F1, macro-F1 and accuracy, with the geometry arm minus
the baseline and a 95% paired bootstrap confidence interval (2,000 resamples of the test images). An interval
entirely above zero is a clear improvement; one that contains zero is no clear change. It needs both arms'
prediction files, from this run's output or attached as inputs.

In [ ]:
# =================================
# Ablation: image only (baseline) vs image + face geometry
# =================================
def find_predictions(arm, part):
    """This run's file first, then any attached output (each arm can run in its own session)."""
    candidates = [os.path.join(WORK_ROOT, f"predictions_{arm}_{part}.npz")]
    candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, "**", f"predictions_{arm}_{part}.npz"), recursive=True))
    for path in candidates:
        if os.path.exists(path):
            return dict(np.load(path)), path
    return None, None


def per_class_f1(y, pred, n_classes):
    """F1 and support per class, from the confusion matrix."""
    cm = np.bincount(y * n_classes + pred, minlength=n_classes * n_classes).reshape(n_classes, n_classes)
    tp = np.diag(cm).astype(float)
    precision = tp / np.maximum(cm.sum(axis=0), 1)
    recall = tp / np.maximum(cm.sum(axis=1), 1)
    return 2 * precision * recall / np.maximum(precision + recall, 1e-12), cm.sum(axis=1)


def summary_scores(y, pred, n_classes):
    """Per-class F1, then macro-F1 over the classes present, then accuracy."""
    f1, support = per_class_f1(y, pred, n_classes)
    return np.concatenate([f1, [f1[support > 0].mean(), (pred == y).mean()]])


def compare_arms(part, n_boot=2000):
    base, base_path = find_predictions("baseline", part)
    geo, geo_path = find_predictions("geometry", part)
    if base is None or geo is None:
        print(f"{part}: needs both predictions_baseline_{part}.npz and predictions_geometry_{part}.npz "
              f"(found baseline: {base_path}, geometry: {geo_path})")
        return None
    assert (base["files"] == geo["files"]).all() and (base["labels"] == geo["labels"]).all(), \
        "The two arms were scored on different images"
    y = base["labels"].astype(int)
    names = [str(n) for n in base["class_names"]]
    n_classes = len(names)
    pred_base, pred_geo = base["probs"].argmax(axis=1), geo["probs"].argmax(axis=1)

    rng = np.random.default_rng(SEED)
    boot = np.empty((n_boot, n_classes + 2))
    for k in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        boot[k] = summary_scores(y[idx], pred_geo[idx], n_classes) - summary_scores(y[idx], pred_base[idx], n_classes)
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5], axis=0)

    score_base, score_geo = summary_scores(y, pred_base, n_classes), summary_scores(y, pred_geo, n_classes)
    rows = names + ["macro-F1", "accuracy"]
    table = pd.DataFrame({"baseline": score_base, "geometry": score_geo, "difference": score_geo - score_base,
                          "ci_low": ci_low, "ci_high": ci_high}, index=rows).round(3)
    table["verdict"] = np.where(ci_low > 0, "better", np.where(ci_high < 0, "worse", "no clear change"))
    print(f"\n===== {part}: {len(y)} images =====\nbaseline: {base_path}\ngeometry: {geo_path}")
    print(table.to_string())

    if geo["face_found"].size:
        found = geo["face_found"].astype(bool)
        print(f"\nFace found in {found.mean():.1%} of these images. Macro-F1 by landmark coverage:")
        for label, mask in (("face found", found), ("no face found", ~found)):
            if mask.sum():
                print(f"  {label:14s} {mask.sum():6d} images | baseline "
                      f"{summary_scores(y[mask], pred_base[mask], n_classes)[n_classes]:.3f} | geometry "
                      f"{summary_scores(y[mask], pred_geo[mask], n_classes)[n_classes]:.3f}")

    diff = score_geo - score_base
    plt.figure(figsize=(10, 4.5))
    plt.errorbar(range(len(rows)), diff, yerr=[diff - ci_low, ci_high - diff], fmt="o", capsize=4)
    plt.axhline(0, color="grey", linewidth=1)
    plt.xticks(range(len(rows)), rows, rotation=30)
    plt.ylabel("geometry minus baseline")
    plt.title(f"Effect of face geometry on {part} ({len(y)} images), with 95% bootstrap CIs")
    plt.tight_layout()
    plt.show()
    return table


ablation_tables = {part: compare_arms(part) for part in ("test", "cross")}